# 07 — Reconstruct the original SMAP gap-filling research table (QA-first)

**Objective:** Rebuild the missing original pixel-date table using the actual three-district CHT boundary. Target: grid-posted SMAP enhanced **AM soil-moisture retrieval**, subject to its retrieval-quality flag; predictors: prior Sentinel-1 radar, prior strict-QA vegetation EVI, CHIRPS rainfall, terrain and calendar information.

This is a **satellite-product reconstruction pilot**, not field-validated root-zone moisture or forest soil moisture. A 9-km SMAP grid posting **does not imply a 9-km independent measurement footprint**. Earth Engine ingests a geographically transformed version of the SMAP data; its reference image projection is **not a claim about original instrument-native EASE-grid geometry**. Predictor values are sampled at grid centers (not footprint-averaged). Expect spatial mismatch in hilly forests. We preserve nonobservations and diagnose gaps; we **do not** replace them with plausible-looking target values.

Run with your local Jupyter kernel authenticated to Earth Engine project `axial-paratext-491911-b6`. Exports are asynchronous Google Drive tasks, avoiding large direct downloads. Source dataset: `NASA/SMAP/SPL3SMP_E/005` (historical 2021 data). Do not substitute `/006` for 2021.


## 1. Local boundary, permissions and pilot controls

The first export is **2021 January–March only**, in one task. After validating the downloaded CSV, change `RUN_FULL_EXPORTS` and select more quarters (ideally several years for a genuinely held-out temporal test). The notebook never uploads your source shapefile to a public repository.


In [18]:
from pathlib import Path
from shapely.geometry import mapping
import hashlib, json
import geopandas as gpd
import pandas as pd
import numpy as np
import ee

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / 'configs' / 'study_config.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from the project root or notebooks/.')
CFG = json.loads((ROOT / 'configs' / 'study_config.json').read_text(encoding='utf-8'))
assert CFG['gee_project_id'] == 'axial-paratext-491911-b6'
boundary = ROOT / CFG['boundary_relative_path']
if not boundary.exists():
    raise FileNotFoundError(f'Boundary shapefile missing: {boundary}')
districts = gpd.read_file(boundary)
assert districts.crs is not None, 'Boundary CRS missing; identify it before use.'
assert districts.geometry.notna().all() and districts.geometry.is_valid.all()
assert len(districts) == 3
assert set(districts[CFG['district_name_field']].astype(str)) == set(CFG['expected_districts'])
districts = districts.to_crs('EPSG:4326')

ee.Initialize(project=CFG['gee_project_id'])  # Must precede ALL ee.Geometry/Image constructors.
features = [ee.Feature(ee.Geometry(mapping(row.geometry.simplify(0.0005, preserve_topology=True))),
                       {'district': str(row[CFG['district_name_field']])})
            for _, row in districts.iterrows()]
DISTRICT_FC = ee.FeatureCollection(features)
AOI = DISTRICT_FC.geometry()

RUN_FULL_EXPORTS = True
RUN_FULL_EXPORTS = True
FULL_YEARS = [2019, 2020, 2021, 2022]
QUARTER_SELECTION = [4, 5]
MAX_TASKS_THIS_RUN = 2
DRIVE_FOLDER = 'CHT_smap_QA_exports'
RAW_DIR = ROOT / 'data' / 'raw' / '07_smap_ee_exports'
PROCESSED_DIR = ROOT / 'data' / 'processed'
TABLE_DIR = ROOT / 'outputs' / 'tables'

def full_quarters(years):
    return [(f'{year}_Q{q}', f'{year}-{first:02d}-01',
             f'{year + (q == 4)}-{(first + 3 - 1) % 12 + 1:02d}-01')
            for year in years for q, first in enumerate((1, 4, 7, 10), start=1)]

periods = full_quarters(FULL_YEARS) if RUN_FULL_EXPORTS else [
    ('2021_Q1_pilot', '2021-01-01', '2021-04-01')]
if QUARTER_SELECTION is not None:
    periods = [periods[i] for i in QUARTER_SELECTION]
periods = periods[:MAX_TASKS_THIS_RUN]
print('Earth Engine project:', CFG['gee_project_id'])
print('Boundary districts:', list(districts[CFG['district_name_field']]))
print('Export periods this run:', periods)


Earth Engine project: axial-paratext-491911-b6
Boundary districts: ['Bandarban', 'Khagrachhari', 'Rangamati']
Export periods this run: [('2020_Q1', '2020-01-01', '2020-04-01'), ('2020_Q2', '2020-04-01', '2020-07-01')]


## 2. Native SMAP grid and predictor definitions

**Important:** SMAP QA `retrieval_qual_flag_am == 0` is a deliberately strict acceptance rule. Original target values and QA flags are exported separately so that missingness can be measured. Sentinel-1 GRD values are **dB**: the valid radar contrast is `VH_dB - VV_dB`, never `VH_dB / VV_dB`. **Model predictors use only earlier calendar days**: previous-day and previous-seven-day rainfall, preceding Sentinel-1 scenes and MOD13Q1 composites whose nominal **16-day interval has already ended**. Same-day CHIRPS is exported only for diagnostics and is **excluded from Notebook 08 features** (same-day rainfall could occur after the SMAP AM overpass). S1 and MODIS missingness is retained, not silently filled. MOD13Q1 EVI is scaled by `0.0001` and screened using `SummaryQA == 0`.

Terrain and vegetation predictors are *grid-center samples*, not SMAP-footprint means. Landslide terrain, land cover and radar incidence-angle effects require future refinements if data coverage is sufficient. This extraction is a feasibility-first experiment.


In [2]:
SMAP = (ee.ImageCollection('NASA/SMAP/SPL3SMP_E/005')
        .select(['soil_moisture_am', 'retrieval_qual_flag_am']))
CHIRPS = ee.ImageCollection('UCSB-CHC/CHIRPS/V3/DAILY_RNL').select('precipitation')
S1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(AOI)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .select(['VV', 'VH']))
MODIS = (ee.ImageCollection('MODIS/061/MOD13Q1')
         .filterBounds(AOI).select(['EVI', 'SummaryQA']))
ELEV = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('elevation_m')
SLOPE = ee.Terrain.slope(ELEV).rename('slope_deg')

reference = ee.Image(SMAP.filterDate('2021-01-01', '2021-02-01').first())
GRID_PROJ = reference.select('soil_moisture_am').projection()
GRID_XY = ee.Image.pixelCoordinates(GRID_PROJ).rename(['grid_x', 'grid_y'])
MASKED_S1 = ee.Image.constant([0, 0]).rename(['s1_vv_db', 's1_vh_db']).updateMask(ee.Image.constant(0))
MASKED_EVI = ee.Image.constant(0).rename('evi_strict_qa').updateMask(ee.Image.constant(0))

print('SMAP reference projection:', GRID_PROJ.getInfo())
print('2021 SMAP images:', SMAP.filterDate('2021-01-01', '2022-01-01').size().getInfo())


SMAP reference projection: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.08, 0, -179.99999994985163, 0, -0.08, 85.04450187956549]}
2021 SMAP images: 365


## 3. One export row per SMAP grid posting × date × district

Earth Engine evaluates operations when exporting; **do not call `getInfo()` on the entire table**. Sampling is performed on the reference **Earth Engine-ingested SMAP image projection**, with stable integer `grid_x` and `grid_y`. It is not guaranteed to preserve the original EASE-Grid pixel indices. The exported `-9999` values denote *missing raster pixels*, never moisture = −9999. We separately remove QA-rejected targets locally.


In [7]:
def predictor_image(smap_image):
    sm = ee.Image(smap_image)
    day = ee.Date(sm.get('system:time_start'))
    rain_day = CHIRPS.filterDate(day, day.advance(1, 'day')).sum().rename('rain_day_mm')
    rain_prior1 = CHIRPS.filterDate(day.advance(-1, 'day'), day).sum().rename('rain_prior1_mm')
    rain_prior7 = CHIRPS.filterDate(day.advance(-7, 'day'), day).sum().rename('rain_prior7_mm')

    radar_collection = S1.filterDate(day.advance(-24, 'day'), day)
    radar_count = radar_collection.size()
    radar = ee.Image(ee.Algorithms.If(
        radar_count.gt(0), radar_collection.median().rename(['s1_vv_db', 's1_vh_db']),
        MASKED_S1))
    radar_difference = radar.select('s1_vh_db').subtract(radar.select('s1_vv_db')).rename('s1_vh_minus_vv_db')

    # MOD13Q1 is a 16-day composite; avoid a nominal start date that could
    # include imagery acquired after this SMAP AM retrieval.
    evi_collection = MODIS.filterDate(day.advance(-48, 'day'), day.advance(-15, 'day'))
    evi_count = evi_collection.size()
    evi = ee.Image(ee.Algorithms.If(
        evi_count.gt(0),
        ee.Image(evi_collection.sort('system:time_start', False).first())
          .select('EVI').multiply(0.0001).rename('evi_strict_qa')
          .updateMask(ee.Image(evi_collection.sort('system:time_start', False).first())
                      .select('SummaryQA').eq(0)),
        MASKED_EVI))

    return ee.Image.cat([
        GRID_XY, sm.select('soil_moisture_am').rename('smap_sm_m3m3'),
        sm.select('retrieval_qual_flag_am'), rain_day, rain_prior1, rain_prior7,
        radar, radar_difference, evi, ELEV, SLOPE,
        ee.Image.constant(radar_count).rename('s1_scene_count_prior24d'),
        ee.Image.constant(evi_count).rename('evi_scene_count_prior48to16d')
    ]).toFloat().unmask(-9999)

EXPORT_FIELDS = [
    'district', 'date', 'grid_x', 'grid_y', 'smap_sm_m3m3',
    'retrieval_qual_flag_am', 'rain_day_mm', 'rain_prior1_mm', 'rain_prior7_mm',
    's1_vv_db', 's1_vh_db', 's1_vh_minus_vv_db',
    'evi_strict_qa', 'elevation_m', 'slope_deg',
    's1_scene_count_prior24d', 'evi_scene_count_prior48to16d'
]

def one_day_features(smap_image, region_feature):
    sm = ee.Image(smap_image)
    district = ee.Feature(region_feature)
    day_text = ee.Date(sm.get('system:time_start')).format('YYYY-MM-dd')
    return predictor_image(sm).sample(
        region=district.geometry(), projection=GRID_PROJ,
        dropNulls=False, geometries=False, tileScale=4
    ).map(lambda f: ee.Feature(f).set({'district': district.get('district'),
                                      'date': day_text}))

def export_table_for_period(start, end):
    images = SMAP.filterBounds(AOI).filterDate(start, end).sort('system:time_start')
    length = images.size()
    days = images.toList(length)

    def samples_for_image(i):
        image = ee.Image(days.get(i))
        all_districts = DISTRICT_FC.toList(DISTRICT_FC.size())
        district_lists = ee.List.sequence(0, DISTRICT_FC.size().subtract(1)).map(
            lambda j: one_day_features(image, ee.Feature(all_districts.get(j))))
        return ee.FeatureCollection(district_lists).flatten()

    all_dates = ee.List.sequence(0, length.subtract(1)).map(samples_for_image)
    return ee.FeatureCollection(all_dates).flatten()


In [8]:
print("Checking Notebook 07 — Section 3")

print("1. Predictor function:", callable(predictor_image))

print("2. Daily sampling function:", callable(one_day_features))

print("3. Export function:", callable(export_table_for_period))

print("4. Number of export fields:", len(EXPORT_FIELDS))

print("\nSection 3 initialized successfully!")

Checking Notebook 07 — Section 3
1. Predictor function: True
2. Daily sampling function: True
3. Export function: True
4. Number of export fields: 17

Section 3 initialized successfully!


## 4. Start Drive export task(s)

Check the printed image counts, then set `START_TASKS=True` and execute once to submit the tasks. Track progress in [Earth Engine Tasks](https://code.earthengine.google.com/tasks), download CSVs from the named Google Drive folder, and copy them into `data/raw/07_smap_ee_exports/`. **Never repeatedly execute the start cell without checking existing tasks**. Some exports may fail because of computation or size; narrow to monthly periods if needed, without dropping QA checks.


In [19]:
from datetime import datetime, timezone

for name, start, end in periods:
    n = SMAP.filterBounds(AOI).filterDate(start, end).size().getInfo()
    print(name, 'SMAP images:', n)
    if n == 0:
        raise ValueError(f'No SMAP images in {name}; adjust dates/source before export.')

START_TASKS = True  # Inspect periods and counts, then change to True explicitly.
if START_TASKS:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    tasks_started = []
    for name, start, end in periods:
        prefix = f'07_cht_smap_{name}'
        task = ee.batch.Export.table.toDrive(
            collection=export_table_for_period(start, end),
            description=prefix, folder=DRIVE_FOLDER,
            fileNamePrefix=prefix, fileFormat='CSV', selectors=EXPORT_FIELDS)
        task.start()
        tasks_started.append({'name': name, 'start': start, 'end_exclusive': end,
                              'task_id': task.id, 'description': prefix})
        print('Submitted:', prefix, task.id)
    receipt = {'created_utc': datetime.now(timezone.utc).isoformat(),
               'gee_project': CFG['gee_project_id'],
               'product': 'NASA/SMAP/SPL3SMP_E/005',
               'qa_rule': 'retrieval_qual_flag_am == 0 AND valid target',
               'predictor_support': 'prior 24-day S1, already-completed 16-day MOD13Q1 EVI, prior-day/7-day CHIRPS; same-day rainfall diagnostic only',
               'tasks': tasks_started}
    outfile = TABLE_DIR / '07_latest_export_receipt.json'
    outfile.write_text(json.dumps(receipt, indent=2), encoding='utf-8')
    print('Task receipt:', outfile)
else:
    print('DRY RUN: set START_TASKS=True to submit Earth Engine export jobs.')


2020_Q1 SMAP images: 91
2020_Q2 SMAP images: 91
Submitted: 07_cht_smap_2020_Q1 KAGBXQ423KA7BCO7OHKCVYT5
Submitted: 07_cht_smap_2020_Q2 SCKJKW4ZWS57VLFJP5ZP4RES
Task receipt: D:\Gihub_Projects\cht-forest-ecohydrology-ml-starter\cht-forest-ecohydrology-ml\outputs\tables\07_latest_export_receipt.json


## 5. Inspect and finalize downloaded exports (CPU only)

You may rerun this cell **later without Earth Engine** (provided earlier initialization cells have already completed). Download every successful `07_cht_smap_*.csv` and copy them into `data/raw/07_smap_ee_exports/`. The finalizer performs duplicate-date/grid checks, flags QA failures, assesses distinct SMAP observation dates and sites, and saves `data/processed/07_smap_ml_table.csv` for Notebook 08. **QA-rejected or missing target rows stay in the table**, and training filters only `target_qa_valid == True`.


In [20]:
import matplotlib.pyplot as plt

files = sorted(RAW_DIR.glob('07_cht_smap_*.csv'))
if not files:
    print('Waiting for Drive CSV files in:', RAW_DIR)
    print('Do not run Notebook 08 yet. No observations will be simulated.')
else:
    frames = []
    for file in files:
        frame = pd.read_csv(file)
        missing = set(EXPORT_FIELDS) - set(frame.columns)
        if missing:
            raise ValueError(f'{file.name} is missing fields: {sorted(missing)}')
        frame = frame[EXPORT_FIELDS].copy()
        frames.append(frame)
        print(file.name, len(frame), 'rows')
    table = pd.concat(frames, ignore_index=True)
    table['date'] = pd.to_datetime(table['date'], format='%Y-%m-%d', errors='raise')
    numeric = [c for c in EXPORT_FIELDS if c not in ('district', 'date')]
    for col in numeric:
        table[col] = pd.to_numeric(table[col], errors='coerce')
        table.loc[table[col] <= -9998, col] = np.nan
    assert set(table.district.unique()).issubset(set(CFG['expected_districts']))
    if table[['date', 'district', 'grid_x', 'grid_y']].isna().any().any():
        raise ValueError('Unusable missing date/district/grid coordinates in export.')
    table['grid_x'] = table['grid_x'].astype('int64')
    table['grid_y'] = table['grid_y'].astype('int64')
    table['grid_id'] = table['grid_x'].astype(str) + '_' + table['grid_y'].astype(str)
    key = ['date', 'grid_id']
    if table.duplicated(key).any():
        examples = table.loc[table.duplicated(key, keep=False),
                             ['date', 'grid_id', 'district']].head(8)
        raise ValueError(f'Duplicate grid postings / dates across exports or districts:\n{examples}')
    table['target_qa_valid'] = (table['retrieval_qual_flag_am'].eq(0)
                                & table['smap_sm_m3m3'].between(0, 1, inclusive='both'))
    table.loc[~table.target_qa_valid, 'smap_sm_m3m3'] = np.nan
    table = table.sort_values(['date', 'district', 'grid_x', 'grid_y']).reset_index(drop=True)

    summary = (table.groupby('district', dropna=False)
               .agg(rows=('target_qa_valid','size'), qa_accepted=('target_qa_valid','sum'),
                    distinct_dates=('date','nunique'), distinct_grids=('grid_id','nunique'))
               .reset_index())
    print('Observed product support (NOT ground accuracy):')
    display(summary)
    supported = table.loc[table.target_qa_valid]
    print('Total accepted pixel-dates:', len(supported),
          '| accepted calendar dates:', supported.date.nunique(),
          '| accepted grids:', supported.grid_id.nunique())
    if supported.empty:
        print('NO QA-accepted SMAP targets. Do not train a model.')

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    destination = PROCESSED_DIR / '07_smap_ml_table.csv'
    table.to_csv(destination, index=False, date_format='%Y-%m-%d')
    summary.to_csv(TABLE_DIR / '07_smap_qa_support_by_district.csv', index=False)
    manifest = {'source_csvs': [{'name': f.name,
                'sha256': hashlib.sha256(f.read_bytes()).hexdigest()} for f in files],
                'target': 'QA accepted enhanced SMAP AM, not in-situ soil moisture',
                'date_min': table.date.min().date().isoformat(),
                'date_max': table.date.max().date().isoformat(),
                'rows': int(len(table)), 'accepted_rows': int(len(supported)),
                'accepted_distinct_dates': int(supported.date.nunique()),
                'accepted_distinct_grids': int(supported.grid_id.nunique())}
    (TABLE_DIR / '07_training_data_manifest.json').write_text(
        json.dumps(manifest, indent=2), encoding='utf-8')
    print('Saved model input:', destination)


07_cht_smap_2019_Q1.csv 16200 rows
07_cht_smap_2019_Q2.csv 14400 rows
07_cht_smap_2020_Q1.csv 16380 rows
07_cht_smap_2020_Q2.csv 16380 rows
07_cht_smap_2021_Q1_pilot.csv 16200 rows
07_cht_smap_2021_Q2.csv 16380 rows
07_cht_smap_2021_Q3.csv 16560 rows
Observed product support (NOT ground accuracy):


,district,rows,qa_accepted,distinct_dates,distinct_grids
0,Bandarban,39375,2624,625,63
1,Khagrachhari,25625,5532,625,41
2,Rangamati,47500,842,625,76


Total accepted pixel-dates: 8998 | accepted calendar dates: 309 | accepted grids: 39
Saved model input: D:\Gihub_Projects\cht-forest-ecohydrology-ml-starter\cht-forest-ecohydrology-ml\data\processed\07_smap_ml_table.csv
